# 20.10 讀懂意思，和逐字抄寫，是同一項能力嗎？


告示寫「今天去臺北」，助理回「今天去台北」。日常對話可能理解了同一地點；如果要求原樣抄寫，「臺」與「台」仍然不同。OCR是從圖片辨識文字，評分要先決定我們要的是意思，還是圖片裡真正的字形。

前置是[11.16的逐字核對](https://birdhackor.github.io/tiny-perceptron-vlm/11.16.html)。整串相同是一把很嚴格的尺，只要有一字多、少或不同就不相同。字元錯誤率CER則多告訴我們改了多少：把預測改成參考文字，最少需要幾次插入、刪除或替換，再除以參考字元數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.natural_concepts import text_error_report

report = text_error_report("今天去臺北", "今天去台北")
print("完整相同", report["exact"])
print("最少編輯次數", report["edits"])
print("參考字數", report["reference_characters"])
print("CER", report["cer"])

輸出是False、1、5與0.2。第一個字串是參考，第二個是預測；只需一次替換，所以是1／5。這裡按Unicode字元計，不按UTF-8儲存bytes計。預測少了最後的「北」，分母仍是參考的5字，不能因漏字改小分母。

文字比較前，有時會整理全形字或空白，這叫正規化。例如NFKC能把全形「Ａ」整理成「A」，卻不把「臺」改成「台」。是否接受異體字、換簡繁、忽略標點，要在看結果前指定；原樣結果也應保留。逐字任務中，即使要求用繁中解釋，圖片寫簡體的部分仍應原樣轉寫，不自行改字。

「沒有字」也需要題目。空白圖若被補成一句熟悉的中文，是多添了不存在的內容。參考空字串時CER沒有可除的字數，因此有字／無字另測；看見模糊字影但讀不清，則是第三種情況，不等於圖片完全沒字。

合成文件讓我們控制字體與答案，真實招牌另外有斜拍、反光與背景。要分開列結果，才能知道模型會讀哪種材料；一張裁切上的短行也不等於完整大頁。新成品的範圍留在[能力卡](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)。練習把預測改成「今天去臺」，再指出刪掉哪一字、CER分母是什麼。字元內容核對完，下一節還有順序。
